<a href="https://colab.research.google.com/github/Liran-Martfel/Project-Vision-CNN_-in-progress/blob/Code-updates/PROJECT_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
import zipfile #importing zip actions
from google.colab import drive #library to connect to drive
drive.mount('/content/drive') #connecting with file-path on drive
zipfile_path = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/project_data.zip' #saving the file path under zipfile_path
with zipfile.ZipFile(zipfile_path,'r') as face_exp_data: #from zipfile library, use ZipFile class
  face_exp_data.extractall('/content/data') #extract all data from zipfile
train_data = '/content/data/train'
#since I'm working with drive, i can't use [....] normally, i need to use the path of the files.
test_data = '/content/data/test'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator #importing the library and action to generate images for the cnn so that he can learn the images

# Training data generator with augmentation
train_datagen = ImageDataGenerator (
    rescale = 1./255, # Normalize pixel values to [0,1] — helps speed up training and stabilize gradients
    shear_range = 0.2,  # Apply a slight diagonal transformation (shear) — simulates natural changes in camera angle
    zoom_range = 0.2,  # Apply random zoom-in effect — helps the model recognize objects at different scales
    horizontal_flip = True # Flip images horizontally — helps the model handle symmetry
)
# Testing data generator — only normalization, no augmentation
test_datagen = ImageDataGenerator(rescale=1./255)

In [23]:
training_set = train_datagen.flow_from_directory( #creating the training set which the cnn will train on with that data
    train_data, #the train data from the exctract ziped file
    target_size = (48,48), # the size of the image
    batch_size = 32, # defualt, the number of images that the model sees all at once.
    class_mode = 'categorical',  #useing categorical beacuse every label is becoming a vector, which will help me when i have more then only 2 chcoices, (like emotions)
    color_mode = 'grayscale')

testing_set = test_datagen.flow_from_directory( #creating the testing set which the cnn will test on with that data
    test_data,
    target_size = (48,48),
    batch_size = 32,
    class_mode = 'categorical',
    color_mode = 'grayscale')

Found 28709 images belonging to 7 classes.
Found 7178 images belonging to 7 classes.


In [ ]:
from keras.models import Sequential #importing the empty model (CNN)
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense,Dropout #importing the 5 types layers of the CNN
cnn = Sequential() #creating the empty CNN model

#first layer, 32 filters of 3X3 weights, 48,48,1 is the original size of the photos & 1 for grayscale
cnn.add(Conv2D (32,(3,3),activation = 'relu', input_shape = (48,48,1))) #define the structure of the layer and add it to the CNN model, using relu as activition as defualt & negatives become 0, also, it's "cheap" and train fast
cnn.add(MaxPooling2D(pool_size = (2,2))) #creating a max_pool with small sqers 2 by 2, and saving the hight number in each sqer

#second layer, 64 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (64,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))

#third layer, 128 filters of 3X3 weights, lose resolution but gain more pattern types.
cnn.add(Conv2D (128,(3,3),activation = 'relu'))
cnn.add(MaxPooling2D(pool_size = (2,2)))
cnn.add(Flatten()) # Flatten, turn the feature maps into one long vector for the layers

#creating the dense
cnn.add(Dense(units = 128,activation= 'relu')) #choose 128 neurons to activate under relu function

#creating a dorpout in order to switch neuron off from the output of the layer before him.
cnn.add(Dropout(0.5)) #0.5 is standart

cnn.add(Dense(units = 7,activation= 'softmax')) #choose 7 neuron beacuse i have 7 emotions to show



In [25]:
cnn.summary() #testing and seeing my CNN model is correct by the way i created him

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,847 (1.36 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
from keras.callbacks import EarlyStopping
cnn.compile(optimizer = 'adam', loss = 'categorical_crossentropy',metrics = ['accuracy'])
# stop training when val_loss stops improving for 5 epochs, and keep the best weights
stopper = EarlyStopping(monitor = 'val_loss',patience = 5, restore_best_weights = True)
cnn.fit (x = training_set, validation_data = testing_set, epochs = 50, callbacks = [stopper])


Epoch 1/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 107s 117ms/step - accuracy: 0.2734 - loss: 1.7710 - val_accuracy: 0.3402 - val_loss: 1.6357
Epoch 2/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 95s 105ms/step - accuracy: 0.3617 - loss: 1.6221 - val_accuracy: 0.4128 - val_loss: 1.4912
Epoch 3/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 98s 109ms/step - accuracy: 0.4085 - loss: 1.5208 - val_accuracy: 0.4696 - val_loss: 1.3806
Epoch 4/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 93s 104ms/step - accuracy: 0.4363 - loss: 1.4597 - val_accuracy: 0.4929 - val_loss: 1.3321
Epoch 5/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 99s 110ms/step - accuracy: 0.4612 - loss: 1.4113 - val_accuracy: 0.5120 - val_loss: 1.2902
Epoch 6/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 99s 110ms/step - accuracy: 0.4760 - loss: 1.3771 - val_accuracy: 0.5095 - val_loss: 1.2748
Epoch 7/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 92s 103ms/step - accuracy: 0.4897 - loss: 1.3421 - val_accuracy: 0.5142 - val_loss: 1.2604
Epoch 8/50
898/898 ━━━━━━━━━━━━━━━━━━━━ 90s 101ms/step - accuracy: 0.4944 - loss: 

In [31]:
cnn.summary()
from tensorflow.keras.models import load_model

cnn.save('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model.keras')
cnn_model = load_model('/content/drive/MyDrive/Colab Notebooks/Project - CNN/training_model.keras')

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,067,543 (4.07 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 711,696 (2.71 MB)

In [103]:
from PIL import Image #Python Image Library
import numpy as np

img_test = '/content/drive/MyDrive/Colab Notebooks/Project - CNN/image test_closeup.jpg'
img = Image.open(img_test)
#print(img) #<PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=447x447 at 0x79C75864E900>
img = img.convert('L') # 'L' in PIL is the colors black & white.
img = img.resize((48,48)) # resizing the img to fit the model (48,48)
print(img.size, img.mode)
img = np.array(img) #transforming the img into an array in order to normalize it
img = img / 255 #normaliting the img
img = img.reshape(1,48,48,1)
print(img.shape, img.dtype, img.min(), img.max())

(48, 48) L
(1, 48, 48, 1) float64 0.06274509803921569 0.9803921568627451


In [33]:
cnn_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 46, 46, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 23, 23, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 21, 21, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 10, 10, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 8, 8, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 4, 4, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       262,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │           903 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,067,543 (4.07 MB)

 Trainable params: 355,847 (1.36 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 711,696 (2.71 MB)

In [126]:
import pandas as pd
prediction = cnn_model.predict(img) #print(dir(training_set))
#print (training_set.class_indices) so i can see the emotions order
#prediction_class_name = training_set.class_indices.keys() ,checking for the keys
prediction_class_name = ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise'] #creating the list of emotions so it can work on the website without the training data
prediction_output = np.argmax(prediction) #taking the place of the highest propability
full_prediction = prediction_class_name[prediction_output] #inserting the data into a new verubale to not get confused
if prediction[0][prediction_output] < 0.6:
  print(f"Unknown, I'm only b{prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}")
else:
  print(f"I'm {prediction[0][prediction_output] * 100:.2f}% sure that this emotion is: {full_prediction}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step
I'm 99.97% sure that this emotion is: happy


In [ ]:
eus